Python ships with a comprehensive standard library — often called "batteries included". This notebook covers the modules you will reach for most often in real projects: `collections` for specialised containers, `itertools` and `functools` for functional utilities, `datetime` for time handling, `re` for pattern matching, `json` for serialisation, `logging` for structured output, `dataclasses` for data containers, `contextlib` for resource management, and `subprocess` for running shell commands.

## `collections` — Specialised Containers

The `collections` module extends Python's built-in `dict`, `list`, and `tuple` with containers optimised for specific patterns.

In [ ]:
from collections import Counter, defaultdict, OrderedDict, namedtuple, deque, ChainMap

# Counter — counts hashable objects
words = ["apple", "banana", "apple", "cherry", "banana", "apple"]
c = Counter(words)
print(c)                          # Counter({'apple': 3, 'banana': 2, 'cherry': 1})
print(c.most_common(2))           # [('apple', 3), ('banana', 2)]
print(c["apple"])                 # 3
print(c["mango"])                 # 0  — missing keys return 0, not KeyError

# Counter arithmetic
c2 = Counter(["apple", "mango", "apple"])
print(c + c2)                     # add counts
print(c - c2)                     # subtract (keeps only positive counts)
print(c & c2)                     # intersection: min of each
print(c | c2)                     # union: max of each

# Count characters
print(Counter("abracadabra"))

In [ ]:
# defaultdict — dict with a factory for missing keys
# No more KeyError when a key is absent

# Group words by first letter
by_letter = defaultdict(list)
for word in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    by_letter[word[0]].append(word)   # list() called automatically for new keys

print(dict(by_letter))

# Count with defaultdict(int)
freq = defaultdict(int)
for ch in "hello world":
    freq[ch] += 1
print(sorted(freq.items()))

# Nested defaultdict
nested = defaultdict(lambda: defaultdict(int))
nested["london"]["rain"] += 1
nested["london"]["sun"]  += 3
nested["paris"]["sun"]   += 5
print({k: dict(v) for k, v in nested.items()})

In [ ]:
# namedtuple — lightweight immutable record with named fields
Point = namedtuple("Point", ["x", "y"])
Color = namedtuple("Color", "red green blue")  # space-separated string also works

p = Point(3, 4)
print(p.x, p.y)          # attribute access
print(p[0], p[1])        # index access — it is still a tuple
print(p._asdict())       # {'x': 3, 'y': 4}

# Replace creates a new instance with changed fields
p2 = p._replace(y=10)
print(p2)                # Point(x=3, y=10)

c = Color(255, 128, 0)
print(c)                 # Color(red=255, green=128, blue=0)

# Useful for returning multiple values with clear semantics
def bounding_box(points: list[Point]):
    BBox = namedtuple("BBox", "min_x max_x min_y max_y")
    xs = [p.x for p in points]
    ys = [p.y for p in points]
    return BBox(min(xs), max(xs), min(ys), max(ys))

pts = [Point(1, 2), Point(5, 3), Point(2, 8)]
bb = bounding_box(pts)
print(bb)                # BBox(min_x=1, max_x=5, min_y=2, max_y=8)

In [ ]:
# deque — double-ended queue; O(1) append/pop at both ends
# list.insert(0, x) and list.pop(0) are O(n) — deque fixes that

dq = deque([1, 2, 3])
dq.append(4)             # append to right
dq.appendleft(0)         # append to left
print(dq)                # deque([0, 1, 2, 3, 4])

dq.pop()                 # remove from right
dq.popleft()             # remove from left
print(dq)                # deque([1, 2, 3])

dq.rotate(1)             # rotate right by 1
print(dq)                # deque([3, 1, 2])
dq.rotate(-1)            # rotate left
print(dq)                # deque([1, 2, 3])

# maxlen — fixed-size sliding window (oldest elements auto-dropped)
recent = deque(maxlen=3)
for i in range(6):
    recent.append(i)
    print(list(recent))  # keeps only last 3

# ChainMap — view multiple dicts as one without copying
defaults  = {"color": "red",  "size": "medium", "debug": False}
overrides = {"color": "blue", "verbose": True}

config = ChainMap(overrides, defaults)   # overrides searched first
print(config["color"])   # 'blue'  — from overrides
print(config["size"])    # 'medium' — from defaults
print(config["debug"])   # False   — from defaults

## `itertools` — Iterator Building Blocks

`itertools` provides memory-efficient tools for working with iterators. All functions return lazy iterators — they produce values on demand rather than building a full list in memory.

In [ ]:
import itertools

# chain — concatenate multiple iterables without copying
a = [1, 2, 3]
b = [4, 5]
c = [6, 7, 8]
print(list(itertools.chain(a, b, c)))         # [1, 2, 3, 4, 5, 6, 7, 8]
print(list(itertools.chain.from_iterable([a, b, c])))  # same, but takes one iterable of iterables

# islice — lazy slice of an iterator
naturals = itertools.count(1)          # infinite counter: 1, 2, 3, ...
print(list(itertools.islice(naturals, 5)))    # [1, 2, 3, 4, 5]
print(list(itertools.islice(range(100), 2, 10, 3)))  # [2, 5, 8] — (start, stop, step)

# cycle — repeat an iterable indefinitely
days = itertools.cycle(["Mon", "Tue", "Wed", "Thu", "Fri"])
schedule = list(itertools.islice(days, 8))
print(schedule)                        # ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Mon', 'Tue', 'Wed']

# repeat — repeat a value n times (or forever if n is omitted)
print(list(itertools.repeat(0, 5)))    # [0, 0, 0, 0, 0]

In [ ]:
# Combinatoric iterators

# product — Cartesian product (nested loops)
suits  = ["♠", "♥", "♦", "♣"]
values = ["A", "K", "Q", "J"]
court_cards = list(itertools.product(values, suits))
print(court_cards[:6])   # [('A', '♠'), ('A', '♥'), ...]
print(f"{len(court_cards)} court cards")

# Equivalent to: [(v, s) for v in values for s in suits]

# permutations — ordered arrangements
perms = list(itertools.permutations([1, 2, 3]))
print(perms)             # all 6 orderings of [1,2,3]

perms2 = list(itertools.permutations([1, 2, 3], 2))  # length-2 permutations
print(perms2)

# combinations — unordered selections without replacement
combs = list(itertools.combinations([1, 2, 3, 4], 2))
print(combs)             # [(1,2), (1,3), (1,4), (2,3), (2,4), (3,4)]

# combinations_with_replacement — allows repeated elements
combs_r = list(itertools.combinations_with_replacement([1, 2, 3], 2))
print(combs_r)

In [ ]:
# groupby — group consecutive elements that share a key
# IMPORTANT: input must be sorted by the key first
data = [
    {"city": "London", "temp": 15},
    {"city": "London", "temp": 17},
    {"city": "Paris",  "temp": 20},
    {"city": "Paris",  "temp": 22},
    {"city": "Berlin", "temp": 12},
]

data.sort(key=lambda d: d["city"])   # must sort by the same key

for city, group in itertools.groupby(data, key=lambda d: d["city"]):
    temps = [r["temp"] for r in group]
    print(f"{city}: avg {sum(temps)/len(temps):.1f}°C")

# accumulate — running totals (or any binary function)
sales = [100, 200, 150, 300, 250]
print(list(itertools.accumulate(sales)))           # running sum
print(list(itertools.accumulate(sales, max)))      # running max
print(list(itertools.accumulate([1,2,3,4,5], lambda a,b: a*b)))  # running product

# takewhile / dropwhile
nums = [1, 3, 5, 7, 8, 10, 11, 12]
print(list(itertools.takewhile(lambda x: x % 2 != 0, nums)))  # [1, 3, 5, 7] — stop at first even
print(list(itertools.dropwhile(lambda x: x % 2 != 0, nums)))  # [8, 10, 11, 12] — skip until even

# compress — filter by a boolean selector
items    = ["a", "b", "c", "d", "e"]
selector = [1,    0,   1,   0,   1]
print(list(itertools.compress(items, selector)))   # ['a', 'c', 'e']

# pairwise — adjacent pairs (Python 3.10+)
print(list(itertools.pairwise([1, 2, 3, 4, 5])))  # [(1,2),(2,3),(3,4),(4,5)]

## `functools` — Higher-Order Function Utilities

`functools` provides tools for working with functions as first-class values: caching, partial application, and reduction.

In [ ]:
from functools import reduce, partial, lru_cache, cached_property, wraps

# reduce — fold a sequence into a single value
from operator import add, mul

print(reduce(add, [1, 2, 3, 4, 5]))      # 15  — sum
print(reduce(mul, [1, 2, 3, 4, 5]))      # 120 — product
print(reduce(add, [1, 2, 3], 100))       # 106 — 100 is the initial value

# Build a pipeline of functions using reduce
def pipeline(*fns):
    """Apply functions left to right."""
    return lambda x: reduce(lambda v, f: f(v), fns, x)

process = pipeline(
    lambda s: s.strip(),
    lambda s: s.lower(),
    lambda s: s.replace(" ", "_"),
)
print(process("  Hello World  "))        # 'hello_world'

In [ ]:
# partial — create a new function with some arguments pre-filled
import math

# Base conversion helpers
to_binary = partial(int, base=2)
to_hex    = partial(int, base=16)

print(to_binary("1010"))    # 10
print(to_hex("ff"))         # 255

# Logging with a fixed prefix
def log(level, message):
    print(f"[{level}] {message}")

warn  = partial(log, "WARN")
error = partial(log, "ERROR")

warn("disk space low")
error("connection refused")

# partial with keyword arguments
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube   = partial(power, exponent=3)
print(square(5), cube(3))    # 25, 27

In [ ]:
import time as _time

# lru_cache — memoize a function's results (Least Recently Used cache)
@lru_cache(maxsize=None)   # unbounded cache; use a number to limit size
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

start = _time.perf_counter()
print(fib(40))                    # computed once per unique argument
print(f"Time: {_time.perf_counter()-start:.4f}s")
print(fib.cache_info())           # CacheInfo(hits=..., misses=..., maxsize=None, currsize=...)

fib.cache_clear()                 # invalidate the cache

# lru_cache with a size limit — evicts least-recently-used entries
@lru_cache(maxsize=128)
def expensive(n):
    _time.sleep(0.01)
    return n * n

for i in range(5):
    expensive(i % 3)              # repeated calls use cache
print(expensive.cache_info())

In [ ]:
# cached_property — compute once, then store as an instance attribute
class Circle:
    def __init__(self, radius: float):
        self.radius = radius

    @cached_property
    def area(self) -> float:
        print("  computing area...")
        return math.pi * self.radius ** 2

    @cached_property
    def circumference(self) -> float:
        print("  computing circumference...")
        return 2 * math.pi * self.radius

c = Circle(5)
print(c.area)           # computed once
print(c.area)           # retrieved from instance __dict__; no recomputation
print(c.circumference)

# wraps — preserve the wrapped function's metadata in a decorator
def timer(fn):
    @wraps(fn)          # copies __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        start = _time.perf_counter()
        result = fn(*args, **kwargs)
        print(f"{fn.__name__} took {_time.perf_counter()-start:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum integers from 0 to n."""
    return sum(range(n))

print(slow_sum(1_000_000))
print(slow_sum.__name__)   # 'slow_sum' — preserved by @wraps
print(slow_sum.__doc__)    # 'Sum integers from 0 to n.' — preserved

## `datetime` — Dates, Times, and Durations

The `datetime` module provides three primary types: `date` (year, month, day), `time` (hour, minute, second, microsecond), and `datetime` (date and time combined). All are immutable. `timedelta` represents a duration.

In [ ]:
from datetime import date, time, datetime, timedelta, timezone

# date — year, month, day
today = date.today()
d = date(2024, 6, 15)
print(today, d)
print(d.year, d.month, d.day)
print(d.isoformat())                 # '2024-06-15'
print(d.strftime("%d %B %Y"))        # '15 June 2024'
print(d.weekday())                   # 0=Mon ... 6=Sun
print(d.isoweekday())                # 1=Mon ... 7=Sun

# datetime — date + time
dt = datetime(2024, 6, 15, 14, 30, 0)
now = datetime.now()                 # local time, no timezone info
utc_now = datetime.now(timezone.utc) # timezone-aware UTC
print(dt)
print(now)
print(utc_now)

In [ ]:
# timedelta — duration
one_week   = timedelta(weeks=1)
two_hours  = timedelta(hours=2)
ninety_min = timedelta(minutes=90)

print(one_week)               # 7 days, 0:00:00
print(two_hours + ninety_min) # 3:30:00

# Arithmetic with dates and datetimes
d = date(2024, 6, 15)
print(d + timedelta(days=10))         # 2024-06-25
print(d - timedelta(weeks=2))         # 2024-06-01
print((date(2024, 12, 31) - d).days)  # days between two dates

meeting = datetime(2024, 6, 15, 9, 0)
reminder = meeting - timedelta(hours=1, minutes=30)
print(f"Reminder at: {reminder}")     # 2024-06-15 07:30:00

# Comparing datetimes
a = datetime(2024, 1, 1)
b = datetime(2024, 6, 1)
print(a < b)                          # True
print(max(a, b))                      # 2024-06-01 00:00:00

In [ ]:
# Parsing and formatting
# strptime — parse a string to datetime
dt = datetime.strptime("15/06/2024 14:30", "%d/%m/%Y %H:%M")
print(dt)

# fromisoformat — parse ISO 8601 strings (Python 3.7+)
dt2 = datetime.fromisoformat("2024-06-15T14:30:00")
print(dt2)

# strftime — format to string
print(dt.strftime("%A, %d %B %Y at %I:%M %p"))  # 'Saturday, 15 June 2024 at 02:30 PM'
print(dt.isoformat())                             # '2024-06-15T14:30:00'

# Common format codes:
# %Y=year  %m=month(01-12)  %d=day  %H=hour(00-23)  %M=minute  %S=second
# %A=weekday name  %B=month name  %I=hour(01-12)  %p=AM/PM

# Timezone-aware datetimes
utc = timezone.utc
eastern = timezone(timedelta(hours=-5), name="EST")

dt_utc = datetime(2024, 6, 15, 14, 30, tzinfo=utc)
dt_est = dt_utc.astimezone(eastern)
print(dt_utc)   # 2024-06-15 14:30:00+00:00
print(dt_est)   # 2024-06-15 09:30:00-05:00

# Timestamps — seconds since Unix epoch
ts = dt_utc.timestamp()
print(ts)
print(datetime.fromtimestamp(ts, tz=utc))

## `re` — Regular Expressions

The `re` module provides Perl-compatible regular expressions. Compile patterns with `re.compile` for reuse, or call module-level functions (`re.match`, `re.search`, `re.findall`, `re.sub`) for one-off uses.

In [ ]:
import re

# match — checks from the START of the string only
# search — scans the ENTIRE string
# fullmatch — the ENTIRE string must match

pattern = r"\d+"

m = re.match(r"\d+", "123 abc")
print(m.group())        # '123'

m = re.match(r"\d+", "abc 123")
print(m)                # None — no digits at start

m = re.search(r"\d+", "abc 123 def")
print(m.group(), m.start(), m.end())   # '123' 4 7

# findall — all non-overlapping matches
print(re.findall(r"\d+", "order 42 has 3 items at price 19.99"))
# ['42', '3', '19', '99']

# finditer — iterator of match objects (memory efficient for large text)
for m in re.finditer(r"\b\w{5}\b", "hello world python regex"):
    print(m.group(), m.span())

In [ ]:
# Groups — extract parts of a match
DATE_RE = re.compile(
    r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})"
)

m = DATE_RE.search("Event on 2024-06-15 at noon")
if m:
    print(m.group(0))              # '2024-06-15'  — full match
    print(m.group(1))              # '2024'
    print(m.group("year"))         # '2024'  — named group
    print(m.groupdict())           # {'year': '2024', 'month': '06', 'day': '15'}

# Multiple matches with named groups
text = "Meetings: 2024-01-10 and 2024-03-22"
for m in DATE_RE.finditer(text):
    d = m.groupdict()
    print(f"{d['day']}/{d['month']}/{d['year']}")

In [ ]:
# sub — replace matches
text = "Call 555-1234 or 555-5678 for info."
print(re.sub(r"\d{3}-\d{4}", "[REDACTED]", text))
# 'Call [REDACTED] or [REDACTED] for info.'

# sub with a function — process each match
def double_digits(m):
    return str(int(m.group()) * 2)

print(re.sub(r"\d+", double_digits, "order 5 qty 3"))  # 'order 10 qty 6'

# Using backreferences in the replacement string
# \1 refers to the first captured group
print(re.sub(r"(\w+)@(\w+)\.com", r"\1 at \2", "alice@example.com bob@test.com"))
# 'alice at example bob at test'

# split — split on a regex pattern
print(re.split(r"[,;\s]+", "one,  two; three  four"))
# ['one', 'two', 'three', 'four']

# Flags — modify matching behaviour
print(re.findall(r"python", "Python PYTHON python", re.IGNORECASE))
# ['Python', 'PYTHON', 'python']

# MULTILINE — ^ and $ match start/end of each line
text = "first line\nsecond line\nthird line"
print(re.findall(r"^\w+", text, re.MULTILINE))
# ['first', 'second', 'third']

## `json` — Serialisation and Deserialisation

`json` converts Python objects to JSON strings and back. The four key functions are `dumps` (to string), `loads` (from string), `dump` (to file), and `load` (from file).

In [ ]:
import json
from datetime import datetime, date

data = {
    "name": "Alice",
    "age": 30,
    "active": True,
    "scores": [95, 87, 92],
    "address": {"city": "London", "postcode": "EC1A"},
    "notes": None,
}

# dumps — Python object → JSON string
compact = json.dumps(data)
pretty  = json.dumps(data, indent=2, sort_keys=True)

print(compact)
print(pretty)

# loads — JSON string → Python object
recovered = json.loads(compact)
print(type(recovered))          # dict
print(recovered["scores"])      # [95, 87, 92]

# Type mapping: JSON null → None, true/false → True/False
print(json.loads('{"x": null, "y": true, "z": false}'))

In [ ]:
import io

# dump / load — serialise to/from a file object
buffer = io.StringIO()
json.dump(data, buffer, indent=2)
buffer.seek(0)
loaded = json.load(buffer)
print(loaded["name"])           # 'Alice'

# Custom encoder — handle types json doesn't know about (e.g. datetime)
class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (datetime, date)):
            return obj.isoformat()
        return super().default(obj)   # raises TypeError for unknown types

event = {"title": "Conference", "date": date(2024, 9, 1)}
print(json.dumps(event, cls=DateTimeEncoder))
# {"title": "Conference", "date": "2024-09-01"}

# Alternatively, use the default= parameter for a simple conversion function
print(json.dumps(event, default=lambda o: o.isoformat() if isinstance(o, (datetime, date)) else str(o)))

# Custom decoder — convert ISO date strings back to date objects
def decode_dates(d):
    for k, v in d.items():
        if isinstance(v, str):
            try:
                d[k] = date.fromisoformat(v)
            except ValueError:
                pass
    return d

raw = '{"title": "Conference", "date": "2024-09-01"}'
result = json.loads(raw, object_hook=decode_dates)
print(result["date"], type(result["date"]))

## `logging` — Structured Output

The `logging` module replaces `print` for diagnostic output. It routes messages through a hierarchy of loggers, filters by severity level, and supports multiple output destinations (console, file, network) simultaneously.

In [ ]:
import logging

# Five severity levels, lowest to highest:
# DEBUG < INFO < WARNING < ERROR < CRITICAL

# basicConfig — one-time root logger setup
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(name)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)

logging.debug("debugging detail")      # only shown if level <= DEBUG
logging.info("application started")
logging.warning("disk space low")
logging.error("database connection failed")
logging.critical("out of memory")

In [ ]:
# Named loggers — one per module; organised in a dot-separated hierarchy
logger = logging.getLogger("myapp.database")
logger.info("connected to %s", "postgres://localhost/db")   # lazy formatting
logger.warning("slow query: %.2fms", 450.3)

# Log exceptions with traceback
try:
    result = 1 / 0
except ZeroDivisionError:
    logger.exception("Division failed")  # logs ERROR + full traceback automatically

# Programmatic handler setup — separate file for errors
app_logger = logging.getLogger("myapp")
app_logger.setLevel(logging.DEBUG)

# Console handler — show INFO and above
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(levelname)-8s %(name)s — %(message)s"))

# File handler — capture DEBUG and above to a log file
fh = logging.FileHandler("/tmp/myapp.log")
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s — %(message)s"))

app_logger.addHandler(ch)
app_logger.addHandler(fh)

app_logger.debug("verbose detail")     # goes to file only
app_logger.info("user logged in")      # goes to both
app_logger.error("payment failed")     # goes to both

print("Log file written to /tmp/myapp.log")

## `dataclasses` — Data Containers Without Boilerplate

The `@dataclass` decorator auto-generates `__init__`, `__repr__`, `__eq__`, and optionally `__hash__`, `__lt__`, and more — eliminating repetitive boilerplate for classes that primarily store data.

In [ ]:
from dataclasses import dataclass, field, asdict, astuple, replace
from typing import ClassVar

@dataclass
class Point:
    x: float
    y: float

    def distance_from_origin(self) -> float:
        return (self.x**2 + self.y**2) ** 0.5

p1 = Point(3.0, 4.0)
p2 = Point(3.0, 4.0)
p3 = Point(1.0, 2.0)

print(p1)               # Point(x=3.0, y=4.0)       — __repr__ generated
print(p1 == p2)         # True                       — __eq__ compares fields
print(p1 == p3)         # False
print(p1.distance_from_origin())   # 5.0

# field() — fine-grained control over individual fields
@dataclass
class Config:
    name: str
    debug: bool = False                       # default value
    tags: list[str] = field(default_factory=list)   # mutable default: must use factory
    _internal: int = field(default=0, repr=False, compare=False)  # excluded from repr and eq

    # Class variable — not a field, not included in __init__
    MAX_TAGS: ClassVar[int] = 10

c = Config("prod", debug=True, tags=["web", "api"])
print(c)

In [ ]:
# __post_init__ — run code after the generated __init__
@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)     # not a constructor parameter

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Dimensions must be positive")
        self.area = self.width * self.height

r = Rectangle(4.0, 5.0)
print(r)               # Rectangle(width=4.0, height=5.0, area=20.0)

try:
    Rectangle(-1, 5)
except ValueError as e:
    print(e)

# frozen=True — immutable; generates __hash__; fields cannot be reassigned
@dataclass(frozen=True)
class Colour:
    r: int
    g: int
    b: int

red = Colour(255, 0, 0)
print(hash(red))        # hashable because frozen
colour_set = {red, Colour(0, 255, 0), red}   # usable as dict key / set member
print(len(colour_set))  # 2

try:
    red.r = 100         # raises FrozenInstanceError
except Exception as e:
    print(e)

# order=True — generates __lt__, __le__, __gt__, __ge__ based on field order
@dataclass(order=True)
class Version:
    major: int
    minor: int
    patch: int

versions = [Version(1, 10, 0), Version(2, 0, 1), Version(1, 9, 3)]
print(sorted(versions))

# Utility functions
print(asdict(r))        # {'width': 4.0, 'height': 5.0, 'area': 20.0}
print(astuple(r))       # (4.0, 5.0, 20.0)
r2 = replace(r, width=10.0)   # like namedtuple._replace — returns new instance
print(r2)

## `contextlib` — Context Manager Utilities

`contextlib` provides tools for creating and working with context managers — including a decorator that turns a generator into a context manager, and helpers for suppressing exceptions and composing multiple managers.

In [ ]:
from contextlib import contextmanager, suppress, nullcontext, ExitStack
import time

# @contextmanager — turn a generator into a context manager
@contextmanager
def timer(label: str):
    start = time.perf_counter()
    try:
        yield                       # body of the with block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label}: {elapsed:.4f}s")

with timer("sorting 100k items"):
    data = sorted(range(100_000, 0, -1))

@contextmanager
def managed_connection(url: str):
    print(f"Connecting to {url}")
    conn = {"url": url, "open": True}
    try:
        yield conn
    except Exception as e:
        print(f"Error during connection: {e}")
        raise
    finally:
        conn["open"] = False
        print(f"Disconnected from {url}")

with managed_connection("postgres://localhost/db") as conn:
    print(f"Connected: {conn}")

In [ ]:
# suppress — silently ignore specific exceptions
import os

# Without suppress:
try:
    os.remove("/tmp/nonexistent_file_xyz.tmp")
except FileNotFoundError:
    pass  # ignore if file doesn't exist

# With suppress — cleaner:
with suppress(FileNotFoundError):
    os.remove("/tmp/nonexistent_file_xyz.tmp")

print("No error — FileNotFoundError suppressed")

# suppress multiple exception types
with suppress(FileNotFoundError, PermissionError):
    os.remove("/tmp/protected_file.tmp")

# nullcontext — a no-op context manager; useful for optional context managers
def process(data, lock=None):
    ctx = lock if lock is not None else nullcontext()
    with ctx:
        return [x * 2 for x in data]

import threading
lock = threading.Lock()
print(process([1, 2, 3], lock=lock))  # with lock
print(process([1, 2, 3]))             # without lock (nullcontext)

# ExitStack — manage a dynamic number of context managers
files = ["/tmp/test_a.txt", "/tmp/test_b.txt", "/tmp/test_c.txt"]

# Create test files
for f in files:
    with open(f, "w") as fh:
        fh.write(f"content of {f}\n")

with ExitStack() as stack:
    handles = [stack.enter_context(open(f)) for f in files]
    for fh in handles:
        print(fh.readline().strip())
# All files are closed when ExitStack exits, in LIFO order

## `math`, `statistics`, and `random`

Three complementary modules: `math` for mathematical constants and functions, `statistics` for descriptive statistics on datasets, and `random` for random numbers and sampling.

In [ ]:
import math
import statistics
import random

# math — mathematical constants and functions
print(math.pi, math.e, math.tau, math.inf, math.nan)

print(math.sqrt(16))          # 4.0
print(math.pow(2, 10))        # 1024.0  (float)
print(2 ** 10)                # 1024    (int — prefer ** for integers)

print(math.floor(3.7), math.ceil(3.2), math.trunc(3.9))
print(math.factorial(10))     # 3628800
print(math.gcd(48, 18))       # 6
print(math.lcm(12, 18))       # 36  (Python 3.9+)
print(math.isclose(0.1 + 0.2, 0.3))  # True — floating-point equality with tolerance
print(math.isinf(math.inf))   # True
print(math.isnan(math.nan))   # True

# Trigonometry (arguments in radians)
print(math.sin(math.pi / 2))  # 1.0
print(math.cos(0))             # 1.0
print(math.degrees(math.pi))   # 180.0
print(math.radians(90))        # 1.5707...

# Logarithms
print(math.log(math.e))        # 1.0  — natural log
print(math.log(1024, 2))       # 10.0 — log base 2
print(math.log10(1000))        # 3.0
print(math.log2(256))          # 8.0

In [ ]:
# statistics — descriptive statistics
scores = [85, 92, 78, 95, 88, 72, 91, 84, 89, 76]

print(statistics.mean(scores))           # arithmetic mean
print(statistics.median(scores))         # middle value
print(statistics.mode(scores))           # most common value (raises if no unique mode)
print(statistics.stdev(scores))          # sample standard deviation
print(statistics.variance(scores))       # sample variance
print(statistics.pstdev(scores))         # population standard deviation
print(statistics.quantiles(scores, n=4)) # quartiles [Q1, Q2, Q3]

# Python 3.10+ — correlation and linear regression
x = [1, 2, 3, 4, 5]
y = [2, 4, 5, 4, 5]
print(statistics.correlation(x, y))      # Pearson correlation
slope, intercept = statistics.linear_regression(x, y)
print(f"y = {slope:.2f}x + {intercept:.2f}")

In [ ]:
# random — pseudo-random numbers (not cryptographically secure)
random.seed(42)   # fix seed for reproducibility

print(random.random())              # float in [0.0, 1.0)
print(random.uniform(1.5, 5.5))    # float in [a, b]
print(random.randint(1, 10))       # integer in [a, b] inclusive
print(random.randrange(0, 100, 5)) # random element of range(0, 100, 5)

# Sequences
items = ["a", "b", "c", "d", "e"]
print(random.choice(items))        # pick one at random
print(random.choices(items, k=3))  # pick k with replacement
print(random.sample(items, k=3))   # pick k without replacement

copy = items[:]
random.shuffle(copy)               # in-place shuffle
print(copy)

# Distributions
print(random.gauss(mu=0, sigma=1))         # Gaussian (normal) distribution
print(random.expovariate(lambd=1/5))       # exponential; mean = 1/lambda

# secrets — cryptographically secure randomness (for tokens, passwords)
import secrets
print(secrets.token_hex(16))        # 32-char hex token
print(secrets.token_urlsafe(16))    # URL-safe base64 token
print(secrets.randbelow(100))       # secure random int in [0, n)

## `subprocess` — Running Shell Commands

`subprocess.run` is the recommended high-level interface for launching and waiting on external processes. Use `Popen` directly only when you need to stream output or interact with the process while it runs.

In [ ]:
import subprocess

# subprocess.run — launch a process and wait for it
result = subprocess.run(
    ["echo", "hello from subprocess"],
    capture_output=True,     # capture stdout and stderr
    text=True,               # decode bytes to str using default encoding
)

print(result.returncode)     # 0 — success
print(result.stdout)         # 'hello from subprocess\n'
print(result.stderr)         # '' — nothing on stderr

# check=True — raise CalledProcessError if the process exits non-zero
try:
    subprocess.run(["ls", "/nonexistent"], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print(f"Exit code: {e.returncode}")
    print(f"stderr: {e.stderr.strip()[:80]}")

In [ ]:
# Passing input to the process
result = subprocess.run(
    ["python3", "-c", "import sys; print(sys.stdin.read().upper())"],
    input="hello world",
    capture_output=True,
    text=True,
)
print(result.stdout)   # 'HELLO WORLD\n'

# Run a shell command (use shell=True only for trusted input — risk of injection)
result = subprocess.run(
    "echo $SHELL",
    shell=True,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())

# Timeout — raise TimeoutExpired if the process takes too long
try:
    subprocess.run(["sleep", "5"], timeout=1, check=True)
except subprocess.TimeoutExpired:
    print("Process timed out")

# Environment variables
import os
env = {**os.environ, "MY_VAR": "hello"}
result = subprocess.run(
    ["python3", "-c", "import os; print(os.environ['MY_VAR'])"],
    env=env,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())   # 'hello'

In [ ]:
# Popen — for streaming output or interacting while the process runs
with subprocess.Popen(
    ["python3", "-c",
     "import time\nfor i in range(4):\n  print(i, flush=True)\n  time.sleep(0.1)"],
    stdout=subprocess.PIPE,
    text=True,
) as proc:
    for line in proc.stdout:     # read line by line as they arrive
        print(f"got: {line.strip()}")

# Piping — stdout of one process into stdin of another
# Equivalent to: ps aux | grep python | wc -l
p1 = subprocess.Popen(["ps", "aux"],          stdout=subprocess.PIPE)
p2 = subprocess.Popen(["grep", "python"],     stdin=p1.stdout, stdout=subprocess.PIPE)
p3 = subprocess.Popen(["wc", "-l"],           stdin=p2.stdout, stdout=subprocess.PIPE)

p1.stdout.close()   # allow p1 to receive SIGPIPE if p2 exits
p2.stdout.close()
output, _ = p3.communicate()
print(f"Python processes running: {output.decode().strip()}")

## Summary

- **`collections`** — `Counter` counts hashable items and supports arithmetic. `defaultdict` supplies a factory for missing keys. `namedtuple` creates lightweight immutable records with named fields. `deque` gives O(1) append and pop at both ends; `maxlen` creates a fixed-size sliding window. `ChainMap` views multiple dicts as one without copying.

- **`itertools`** — All functions return lazy iterators. `chain` concatenates iterables. `islice` slices an iterator. `cycle` and `repeat` produce infinite sequences. `product`, `permutations`, and `combinations` generate combinatoric sequences. `groupby` groups consecutive elements (input must be pre-sorted). `accumulate` computes running aggregates. `takewhile` and `dropwhile` filter by a predicate.

- **`functools`** — `reduce` folds a sequence with a binary function. `partial` pre-fills arguments to create specialised callables. `lru_cache` memoises pure functions with a configurable cache size. `cached_property` computes once and stores the result in the instance. `wraps` preserves function metadata in decorators.

- **`datetime`** — `date`, `time`, and `datetime` are immutable. `timedelta` represents a duration; add or subtract it from dates and datetimes. `strptime` parses strings; `strftime` formats them. Use `timezone`-aware datetimes for any code that crosses time zones. `timestamp()` and `fromtimestamp()` convert to and from Unix epoch seconds.

- **`re`** — `match` anchors to the start; `search` scans the whole string; `findall` returns all matches; `sub` replaces them. Named groups `(?P<name>...)` make patterns self-documenting. Compile patterns with `re.compile` when reusing them. Common flags: `IGNORECASE`, `MULTILINE`.

- **`json`** — `dumps`/`loads` convert between Python objects and JSON strings. `dump`/`load` use file objects. Pass `cls=` a custom `JSONEncoder` subclass, or `default=` a function, to handle non-serialisable types. Pass `object_hook=` a function to `loads` to decode custom types on the way back.

- **`logging`** — Five levels: DEBUG < INFO < WARNING < ERROR < CRITICAL. Use `logging.getLogger(__name__)` in each module to create a named logger. Attach `StreamHandler` for console output and `FileHandler` for persistent logs. Log exceptions with `logger.exception()` to capture the traceback automatically. Never use `print` for diagnostic output in library or production code.

- **`dataclasses`** — `@dataclass` generates `__init__`, `__repr__`, and `__eq__`. Use `field(default_factory=...)` for mutable defaults. `__post_init__` runs after the generated initialiser. `frozen=True` makes instances immutable and hashable. `order=True` generates comparison methods. `asdict`, `astuple`, and `replace` are utility functions for conversion and copying.

- **`contextlib`** — `@contextmanager` turns a generator with a single `yield` into a context manager. `suppress(ExcType)` silently swallows specific exceptions. `nullcontext` is a no-op manager useful for optional locking. `ExitStack` manages a dynamic number of context managers.

- **`math`** provides constants (`pi`, `e`, `inf`) and functions (`sqrt`, `floor`, `ceil`, `factorial`, `gcd`, `log`, `sin`). `math.isclose` is the right way to compare floats. `statistics` computes `mean`, `median`, `stdev`, `quantiles`, `correlation`, and `linear_regression`. `random` generates pseudo-random values and supports sampling and shuffling; use `secrets` for cryptographic randomness.

- **`subprocess`** — `subprocess.run` is the standard interface: pass a list of arguments, `capture_output=True` to collect stdout/stderr, `text=True` for string output, `check=True` to raise on non-zero exit. Use `timeout=` to enforce deadlines. Use `Popen` when you need to stream output or pipe processes together.